# Reliable tool use with Claude: validate args and gate egress

When you give Claude tools, two failure modes show up in production that the model itself can't always avoid:

1. **The model hands you tool args that don't quite match your schema** — a missing required field, a string where a number belongs, a malformed URL. By the time the tool throws, you've already burned a turn on a failure the LLM could have self-corrected if you'd told it what was wrong.
2. **The model hands you tool args that match the schema but ask the tool to fetch something you didn't authorize** — an internal hostname, a credential-leaking URL, a domain that wasn't on yesterday's allowlist.

This notebook wires two small guardrails into a `tool_use` loop with Claude:

- **[`agentvet`](https://pypi.org/project/agentvet-py/)** — validates tool args before the tool runs, and emits an LLM-friendly retry hint that you send back as a `tool_result` with `is_error=True`. Claude sees the validation error, fixes its args, and retries on the next turn.
- **[`agentguard`](https://pypi.org/project/agentguard-firewall/)** — a declarative network-egress firewall. URLs that aren't on your allowlist (or that hit your denylist) are blocked before any HTTP request goes out.

Both libraries are pure Python, zero runtime dependencies, MIT licensed. They don't know what an LLM is — you wire them in around your tool loop.

## Step 1: Install + setup

In [ ]:
%pip install -q anthropic agentvet-py agentguard-firewall python-dotenv

In [ ]:
# agentguard: declarative network egress firewall
from agentguard import check, policy

# agentvet: validate tool args before execution
from agentvet import ToolArgError, vet
from agentvet import adapters as vet_adapters
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

## Step 2: Define a tool that fetches web pages

Our example tool is simple: `fetch_url` takes a URL and a max length, returns a snippet of the page. It's the kind of tool that's easy for Claude to call wrong (forget the max_length, pass a non-string URL) and easy to abuse (point at internal hostnames, credential URLs).

In [ ]:
import urllib.request


def _fetch_url_impl(args):
    """The actual tool implementation. Receives validated args."""
    url = args["url"]
    max_length = args.get("max_length", 500)

    req = urllib.request.Request(url, headers={"User-Agent": "agent-stack-demo"})  # noqa: S310
    with urllib.request.urlopen(req, timeout=10) as response:  # noqa: S310
        body = response.read().decode("utf-8", errors="replace")

    return body[:max_length]

## Step 3: Wrap with agentvet for arg validation

`agentvet.vet()` returns a tool function that validates args first. If validation fails it raises `ToolArgError`, which has a `to_llm_feedback()` method that produces a string Claude can read and act on.

The shape adapter is the simplest validator. For richer rules use `adapters.pydantic(MyModel)` or `adapters.fn(predicate)`.

In [ ]:
# Define the schema once, use it in two places: agentvet validation + Claude's tool definition.
URL_TOOL_SHAPE = {
    "url": "str",
    "max_length": "int?",  # ? suffix marks optional
}

vetted_fetch = vet(
    name="fetch_url",
    schema=vet_adapters.shape(URL_TOOL_SHAPE),
    fn=_fetch_url_impl,
)

Quick sanity check that validation fires when we expect:

In [ ]:
# Good args
try:
    out = vetted_fetch({"url": "https://example.com", "max_length": 200})
    print("OK, fetched", len(out), "chars")
except ToolArgError as e:
    print("validation failed:", e.validation_error)

# Bad args: missing required field
try:
    vetted_fetch({"max_length": 200})
except ToolArgError as e:
    print("\ncaught:", e.validation_error)
    print("\nfeedback for Claude:\n", e.to_llm_feedback())

## Step 4: Layer agentguard on top for egress safety

A policy is a plain dict. The most useful field is `network.allow` — a list of host patterns Claude is permitted to reach. Anything not on that list (or anything on `deny`, which wins over `allow`) gets refused before the network request goes out.

Pattern syntax:
- `"example.com"` — exact host
- `"*.example.com"` — example.com plus any subdomain
- `"*"` — match everything (handy as a catch-all in `deny`)

In [ ]:
network_policy = policy(
    {
        "network": {
            "allow": [
                "example.com",
                "*.python.org",
                "api.github.com",
            ],
            "deny": [
                "*.internal",  # don't let Claude probe internal hosts
                "169.254.169.254",  # cloud metadata services
            ],
        },
    }
)

# Check a few URLs
for url in [
    "https://example.com/",
    "https://docs.python.org/3/",
    "https://api.github.com/repos/anthropics/anthropic-sdk-python",
    "https://internal.corp.internal/secrets",
    "https://evil.example.org/",
]:
    decision = check(network_policy, url)
    if decision.action == "allow":
        print(f"  ALLOW  {url}")
    else:
        print(f"  DENY   {url}  ({decision.reason}: {decision.detail})")

## Step 5: Combine agentvet + agentguard in one tool

The tool's flow becomes: validate args → check egress policy → execute. We use `agentvet`'s `on_error` callback to fold the egress check into a single `ToolArgError` so Claude sees one consistent retry pattern for both kinds of rejection.

In [ ]:
def _safe_fetch_impl(args):
    url = args["url"]
    decision = check(network_policy, url)
    if decision.action != "allow":
        # Surface as a ToolArgError so the retry loop is uniform.
        raise ToolArgError(
            "fetch_url",
            f"URL '{url}' is not permitted by the network policy "
            f"(reason: {decision.reason}). Allowed hosts: example.com, *.python.org, "
            f"api.github.com.",
            args,
        )
    return _fetch_url_impl(args)


safe_fetch = vet(
    name="fetch_url",
    schema=vet_adapters.shape(URL_TOOL_SHAPE),
    fn=_safe_fetch_impl,
)

## Step 6: Wire it into a Claude tool_use loop

The tool definition Claude sees is the standard Anthropic shape. The validation + egress firewall live in *our* code, not in Claude's tool schema — Claude doesn't need to know about them.

When a `ToolArgError` fires, we send the feedback string back as a `tool_result` with `is_error=True`. Claude reads it and self-corrects on the next turn.

In [ ]:
tools = [
    {
        "name": "fetch_url",
        "description": (
            "Fetch a snippet of a web page by URL. Use this when the user asks "
            "for the contents of a public web page."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {"type": "string", "description": "Absolute URL to fetch."},
                "max_length": {
                    "type": "integer",
                    "description": "Max characters to return (default 500).",
                },
            },
            "required": ["url"],
        },
    }
]


def run_with_safe_tool(user_message, max_iterations=4):
    """Run a tool_use loop, surfacing validation errors back to Claude."""
    print(f"\n{'=' * 60}\nUser: {user_message}\n{'=' * 60}")
    messages = [{"role": "user", "content": user_message}]

    for _iteration in range(max_iterations):
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # Append Claude's turn (text + any tool_use blocks).
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            # Conversation finished. Print the final text.
            text = "".join(b.text for b in response.content if getattr(b, "type", "") == "text")
            print(f"\nFinal: {text}")
            return text

        # Process every tool_use block in this turn.
        tool_results = []
        for block in response.content:
            if getattr(block, "type", "") != "tool_use":
                continue
            print(f"\n  tool_use: {block.name}({block.input})")
            try:
                result = safe_fetch(block.input)
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result),
                    }
                )
                print(f"    ok ({len(str(result))} chars)")
            except ToolArgError as err:
                # Send the LLM-friendly feedback back as is_error=True.
                feedback = err.to_llm_feedback()
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": feedback,
                        "is_error": True,
                    }
                )
                print(f"    rejected: {err.validation_error}")

        messages.append({"role": "user", "content": tool_results})

    print("\n(hit max_iterations without natural stop)")
    return None

## Step 7: See it in action

Three scenarios:

1. **Happy path** — Claude calls the tool with sensible args against an allowed host.
2. **Egress denial** — Claude tries to reach a host that isn't on the allowlist. The agent gets a clear error and explains it to the user instead of hanging.
3. **Self-correction** — Claude is gently nudged toward a domain that *is* allowed, and after the rejection it picks the correct one.

In [ ]:
# 1. Happy path
run_with_safe_tool("Fetch the homepage at https://example.com and tell me what's on it.")

In [ ]:
# 2. Egress denial — the model wants a host that isn't on the allowlist.
run_with_safe_tool("Fetch https://news.ycombinator.com and summarize the top story.")

In [ ]:
# 3. Self-correction — the user mentions an unallowed URL, but Claude has
#    other allowed hosts available and should pick one of those.
run_with_safe_tool(
    "I want to read about CPython internals. Try https://internal.docs.internal first, "
    "and if that fails, look on docs.python.org."
)

## What this gives you

- **Faster recovery from arg-shape errors.** Instead of a tool throwing a generic exception that ends the run, the validation error becomes part of Claude's context and the next turn fixes it.
- **A real network boundary.** `agentguard` is a declarative allowlist enforced before any request. Even if Claude is prompt-injected by content it just fetched, it can't follow up by reaching a host you didn't authorize.
- **One shape definition, two uses.** The `URL_TOOL_SHAPE` dict drives both `agentvet` validation and (with a small adaptation) Claude's tool input_schema. Update the shape in one place.

## Going further

- **Custom validators**: `vet_adapters.fn(predicate)` for ad-hoc checks; `vet_adapters.pydantic(Model)` for richer types with field-level errors.
- **Methods + budgets**: `policy()` also accepts `network.methods` (e.g. allow `GET` only) and `budget.max_requests` for soft rate-limit enforcement.
- **Ecosystem**: this notebook uses two of the five libraries in the [agent reliability stack](https://mukundakatta.github.io/agent-stack/). The others — `agentfit` (token-budget truncation), `agentsnap` (tool-call trace diffing), `agentcast` (structured-output extraction) — solve the matching problems on the rest of the agent loop.